# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [4]:
print(df.shape)
print()
print(df.dtypes)
print()
print("пропусков в delivery_days:", df["delivery_days"].isna().sum())
print()
print(df[["city", "category", "status", "customer_id"]].nunique())

(5463, 8)

order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object

пропусков в delivery_days: 160

city             14
category          5
status            2
customer_id    2003
dtype: int64


In [5]:
print(df["city"].value_counts())
print()
print(df.loc[df["amount"] == df["amount"].max(), ["order_date", "customer_id", "city", "amount", "status"]])

city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64

            order_date customer_id    city     amount   status
1001  2026-09-12 11:00        TEST  Алматы  9999990.0  оплачен
2268  2026-09-25 15:00        TEST  Алматы  9999990.0  оплачен
2966  2026-10-03 10:00        TEST  Алматы  9999990.0  оплачен


Города записаны разными способами по типу, АЛМАТЫ, Almaty, Алматы и т.д.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
a = df.loc[paid & not_test, "amount"]
print(f"заказов: {len(a)}")
print(f"среднее: {a.mean():,.0f} ₸")
print(f"медиана: {a.median():,.0f} ₸")
print(f"25-й перцентиль: {a.quantile(0.25):,.0f} ₸")
print(f"75-й перцентиль: {a.quantile(0.75):,.0f} ₸")
print(f"максимум: {a.max():,.0f} ₸")

заказов: 4901
среднее: 14,672 ₸
медиана: 10,300 ₸
25-й перцентиль: 5,880 ₸
75-й перцентиль: 18,690 ₸
максимум: 139,040 ₸


типичный заказ 10,300 ₸, в отчете лучше показывать медиану, т.к. она отвечает на вопрос сколько тратит обычный покупатель

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [7]:
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancel = df["status"] == "отменён"

all_share = cancel.mean() * 100
sh_share = cancel[df["city"].isin(SH)].mean() * 100
al_share = cancel[df["city"].isin(AL)].mean() * 100

print(f"все заказы: {all_share:.1f}%")
print(f"Шымкент: {sh_share:.1f}%")
print(f"Алматы: {al_share:.1f}%")
print(f"Шымкент / Алматы: {sh_share / al_share:.1f}")
print(f"Шымкент / все: {sh_share / all_share:.1f}")

все заказы: 10.2%
Шымкент: 20.4%
Алматы: 7.9%
Шымкент / Алматы: 2.6
Шымкент / все: 2.0


в среднем отменяют 10.2% заказов, а в шымкенте отменяется 20.4% заказов

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [8]:
for name, spellings in [("Шымкент", SH), ("Алматы", AL)]:
    m = paid & not_test & df["city"].isin(spellings)
    s = df.loc[m & sep, "amount"].sum()
    o = df.loc[m & octo, "amount"].sum()
    print(f"{name}: сентябрь {s:,.0f} ₸, октябрь {o:,.0f} ₸, изменение {(o - s) / s * 100:+.1f}%")

Шымкент: сентябрь 4,949,800 ₸, октябрь 4,044,010 ₸, изменение -18.3%
Алматы: сентябрь 15,523,940 ₸, октябрь 14,643,020 ₸, изменение -5.7%


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [9]:
try:
    cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
except Exception as e:
    print(type(e).__name__, e)

wrong = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print("только 'Шымкент':", len(wrong))
print("все написания:", len(cancel_sh))

TypeError unsupported operand type(s) for &: 'str' and 'StringArray'
только 'Шымкент': 153
все написания: 164


строку и сериес соединить нельзя поэтому надо скобки
мы ищем =="Шымкент", а у нас в датасете есть еще "шымкент" и "Shymkent" 

**Ошибка 1 (роняет код):** у `&` приоритет выше, чем у `==`. Python сначала пытается вычислить `"Шымкент" & df["status"]`, а строку с Series через `&` соединить нельзя, отсюда TypeError. Каждое условие нужно взять в скобки.

**Ошибка 2 (тихо даёт неверное число):** `== "Шымкент"` находит только одно написание и пропускает «шымкент» и «Shymkent». Получается 153 отмены вместо 164. Нужно `isin(SH)`.

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [10]:
vc = df["customer_id"].value_counts()
print(vc.head())
top = vc.index[0]

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64


In [11]:
orders = df.loc[df["customer_id"] == top, ["order_id", "order_date", "city", "amount", "status"]]
print(orders)
print()
print(orders["order_id"].value_counts().head(3))

      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

order_id
101504    2
101125    1
100096    1
Name: count, dtype: int64


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [12]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [13]:
# 8